In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd /content/drive/MyDrive/IntroAI/Lab7

# Lab 7 Dataset

This part prepares data for developing a convolutional neural network (CNN) using PyTorch for solving a simple classification problem using the MNIST dataset of fashion. It should be used in combination with a CNN model, such as a self-defined CNN, AlexNet, ResNet or a VGGNet.

You need to import the package `torch` in order to use PyTorch and `torchvision` to use methods of dealing with datasets. If you don't have these packages in your environment, you need to install them first using `pip install <package-name>`.

You also need a set of predefined classes which are packed in `heading.py`, such as AlexNet, ResNet and VGG.

In [ ]:
from heading import *
import torch
from torchvision import datasets

### THE MNIST DATABASE OF FASHIONS

Fashion-MNIST is a dataset of Zalando's article images—consisting of a training set of 60,000 examples and a test set of 10,000 examples. Each example is a 28x28 grayscale image, associated with a label from 10 classes.
Link: https://www.kaggle.com/zalando-research/fashionmnist

Labels

Each training and test example is assigned to one of the following labels:

    0 T-shirt/top
    1 Trouser
    2 Pullover
    3 Dress
    4 Coat
    5 Sandal
    6 Shirt
    7 Sneaker
    8 Bag
    9 Ankle boot

It is a good database for people who want to learn image classification methods using real-world data.

Since the input image size is different in different CNN architectures, you need to prepare datasets based on each CNN architecture.

You download the datasets and store them in a sub-folder "Data" in your Lab 8 folder.

### Load MNIST datasets

First you specify the input image size used in a target CNN model.
* If you use the self-defined CNN or the ResNet, the image dimension is 28 * 28, so the image size is 28.
* If you use the AlexNet, the image dimension is 227 * 227, so the image size is 227.
* If you use the VGGNet, the image dimension is 224 * 224, so the image size is 224.

In [ ]:
# For an AlexNet model
image_size = 28

Then, you need to tranform the original image shape to the image size used in the target network by using `resize_image(<image-size>)` method.

In [ ]:
transform = resize_image(image_size)

Divide the dataset into BATCH_SIZE equal parts for training.

## Important message

A CNN variant, such as AlexNet, ResNet or VGGNet, is usually a large scale network. If you train it using the entire dataset, it might cost several hours (on cpu). This lab is mainly for you to learn how to configure and how to train a CNN model, thus, you can set the batch size to be 1 for simplicity. If you want to try with the entire dataset, you can set the batch size to 64.

In [ ]:
BATCH_SIZE = 1

You need to define a `load_data()` method to load training dataset and test dataset based on the BATCH_SIZE.

Here you define this method for loading the dataset Fashion-MNIST. If you want to try other datasets, you can change the dataset name and path to accommodate your new dataset, e.g. `datasets.MNIST`, `datasets.CIFAR10`, or `datasets.CIFAR10`.

In [ ]:
def load_data(transform, BATCH_SIZE, train, PATH = 'Data'):
    load_dataset = datasets.FashionMNIST(PATH,
                               train=train,
                               transform=transform,
                               download=False)

    data_loader = torch.utils.data.DataLoader(dataset = load_dataset,
                                            batch_size = BATCH_SIZE, shuffle = True)

    return data_loader

You then load the training data and test data separately using the `load_data()` method. For the training data, you need to set the argument `train` to be `True`. For the testing data, you need to set the this argument to be `False`. The path is `Data` which is in the current directory.

In [ ]:
train_loader = load_data(transform, BATCH_SIZE, train = True, PATH = 'Data')
test_loader = load_data(transform, BATCH_SIZE, train = False, PATH = 'Data')

You need to convert the labels to object's names. You can create a list that contains all categories in order.

In [ ]:
label_list = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

You can make a future image sample set. You can use the CNN model to predict the labels of these samples.
In order to do that, you need to specify the location of your future image data by using the`FlameSet(image-path, image-size)`, where `image-path` is the path to the folder containing the images and image-size is the image size used in the pre-trained model. In this case, the folder path is Image in the current directory, i.e., ./Image.
The image size should match with the target CNN model.
* If you use the AlexNet, the image dimension is 227 * 227, so the image size is 227.
* If you use the ResNet, the image dimension is 28 * 28, so the image size is 28.
* If you use the VGGNet, the image dimension is 224 * 224, so the image size is 224.


In [ ]:
dataSet = FlameSet('./Image', image_size = 28)

You can check the dataset specified by listing the files in this linked folder.

In [ ]:
dataSet.get_image_list()

Once the image files are confirmed, you can select one image or all to predict the output label(s) using the target CNN model.

In [ ]:
# Select the 4th image file in the specified folder.
select_image = dataSet[3]

### View MNIST datasets

You can view data samples in a batch in the train or test or future sample loader, i.e., you can view BATCH_SIZE images and their labels at each time.  For example, when BATCH_SIZE = 1, you can view the image in the train or test loader as a tensor that represents the image as a 227x227 pixel matrix.

In [ ]:
view_datasets(train_loader, label_list)

In [ ]:
view_datasets(test_loader, label_list)

You can also view your selected image sample using methods in OpenCV.

In [ ]:
import cv2 as cv
from google.colab.patches import cv2_imshow
img=cv.imread('./Image/5-0.png')
cv2_imshow(img)

# Lab 7 Algorithm -- ResNet
In this part, you will learn how to use PyTorch to train a convolutional neural network,

* ResNet,

to solve a given problem using the dataset sepcified above.

You will see three scenarios of developing a convolutional neural network to solve a given problem:
* Scenario 1: Develop a variant of CNN model from scratch, in this case, it is ResNet
* Scenario 2: Use a well trained model which was trained for solving a similar problem
* Scenario 3: Re-train a pre-trained model which was trained for solving a similar problem

### Scenario 1: Develop a ResNet model from scratch
You first configure this variant of convolutional neural network, `ResNet`, then train and test it, lastly apply it to classify future images.  


Now, you can use the constructor of ResNet class to create instances of ResNet by providing the `number-of-basic-block`, `input-channels` and `output-size`. The basic blocks are blocks consisting of convolutional layers and defined in heading.py.

`input-channels` is 1 for gray images and `input-channels` is 3 for colour images. `output-size` is the number of classes.

In this lab case, input-channels is 1 and output-size is 10 for 10 classes of Fashion items to be classified.

In [ ]:
input_channels = 1
output_size = 10

Following is a ResNet instance, ResNet18,  where 18 is the number of layers in the network.

In [ ]:
ResNet18 = ResNet(BasicBlock, [2,2,2,2],input_channels, output_size)

Then, you can create one instance of ResNet by calling the `create_network(network-instance)` method, where `network-instance` is the one defined in the previous step, ResNet18.

In [ ]:
res_net_18 = create_network(ResNet18)

Prepare parameters for training

You need to define the `number of epoches` and the `learning rate`.
You can set the `number of epoches` as 1 for simplicity and the `learning rate` as 0.001.

You also need to specify the `number of images` used in training. If number of images is None, the training will use all the images in the training dataset, otherwise the training will use the specified number of images.

In this case, you can set the number of images as 100. Thus the training will use 100 random images from the training dataset to train the model.

In [ ]:
EPOCH = 1
LR = 0.001
number_of_images = 100

Now, you can train the model by using the method `train_model(target-Network,train-loader, learning-rate, number-of-epoches, number-of-images)`.

In [ ]:
trained_model_resnet18 = train_model(res_net_18, train_loader, LR, EPOCH, number_of_images)

After training, you can test the model using the test dataset by using the method `test_model(trained-model, test-loader, number-of-images)`. You can try 10 images from the test dataset, i.e., number-of-images = 10.

In [ ]:
test_model(trained_model_resnet18, test_loader, number_of_images = 10)

You can see the accuracy values of these models are rather poor as you did not use adequate training data to train them. In solving real world problems, you need to use a large size of training dataset to get a usable model.

Lastly, you can use the method predict_image( trained-model, image-data, objective-list, number-of-prediction) to classify your selected image sample, which is defined in the dataset part.

In [ ]:
predict_image(trained_model_resnet18, select_image, label_list)

If you want to save your model, you can use `torch.save(model, model path and name)` to save your model.

In [ ]:
torch.save(trained_model_resnet18, 'ResNet18_model.pth')

### Scenario 2: Use a well trained ResNet model which was trained for solving a similar problem

#### Important Messages

Since the effort to train a good CNN model is huge, it is common to use a pre-trained model that was trained for solving some similar problems to the given problem.

In this case, you can use a pre-trained model which is available with the lab materal pack, `ResNet18_pretrained_model.pth`.

You need to download this model and save it in your Lab 7 folder.

In order to use this pre-trained model, you need to load it by calling the `load()` method in pytorch, i.e., torch.load().

Pretrained model:

* ResNet18_pretrained_model.pth,      Image dimension is 28x28

If your device does not have CUDA device, please add `map_location=torch.device('cpu')`in the call to the load method, e.g., torch.load(model-name,map_location=torch.device('cpu')).

In [ ]:
pretrained_model_resnet18 = torch.load('ResNet18_pretrained_model.pth', map_location=torch.device('cpu'))

You can test this pretrained model using the test dataset by calling the method test_model().

In [ ]:
 test_model( pretrained_model_resnet18, test_loader, number_of_images = 10)

Lastly, you can use the method predict_image( trained-model, image-data, objective-list, number-of-prediction) to classify your selected image sample.

In [ ]:
predict_image(pretrained_model_resnet18, select_image, label_list)

### Scenario 3: Re-train a pre-trained ResNet model which was trained for solving a similar problem

### Important Messages

Since the effort to train a good CNN model is huge, it is common to use transfer learning to make use of a pre-trained model which was trained for solving some similar problems.
There are a number of ways of using a pre-trained model. Within the scope of this subject, you only use the pre-trained model as a model with better selection of meta-parameters (weights) and re-train it as you train a self-defined CNN.

In this case, you can re-train a pre-trained model which is available with the lab material pack, `ResNet18_pretrained_model.pth`.

You need to download this model and save it in your Lab 7 folder.

In order to access this pre-trained model, you need to load it as you do in Scenario 2.

In [ ]:
pretrained_model_resnet18 = torch.load('ResNet18_pretrained_model.pth', map_location=torch.device('cpu'))

You then can re-train the loaded pre-trained model using the data from the given problem. In this case, you use the training dataset.

In [ ]:
re_trained_model_resnet18 = train_model(pretrained_model_resnet18, train_loader, LR, EPOCH, number_of_images)

After re-training, you can test this re-trained model using the test dataset by using the method `test_model(trained-model, test-loader, number-of-images)`. You can try 10 images from the test dataset, i.e., number-of-images = 10.

In [ ]:
test_model(re_trained_model_resnet18, test_loader, number_of_images = 10)

Lastly, you can use the method predict_image( trained-model, image-data, objective-list) to classify your selected image sample, which is defined in the dataset part.

In [ ]:
predict_image(re_trained_model_resnet18, select_image, label_list)